# County Rainfall and Planting Advisory
**Author:** Chepkirui Joyce Ngetich

**Cohort:** PLP Data Analytics Cohort, 2026
  
**Stack:** Core Python (`csv`, `statistics`), `numpy`

**Dataset & Task 1:** Cleaning machakos_raw

In [5]:
import csv
import statistics
import numpy as np

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]
months = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December",
]

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],  # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],  # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],  # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],  # Mombasa
]

machakos_raw = [
    "48",
    "35",
    "88",
    "N/A",
    "140",
    " 20 ",
    "9",
    "12",
    "",
    "45",
    "182",
    "96",
]


def clean_machakos_series(raw_list, month_names):
  """Convert valid string readings to ints; impute ValueError entries with rounded valid mean."""
  parsed = []
  imputed_months = []

  for month_name, raw_val in zip(month_names, raw_list):
    try:
      parsed.append(int(str(raw_val).strip()))
    except ValueError:
      parsed.append(None)
      imputed_months.append(month_name)

  valid_vals = [v for v in parsed if v is not None]
  imputed_val = round(statistics.mean(valid_vals))
  cleaned_row = [imputed_val if v is None else v for v in parsed]
  return cleaned_row, imputed_months, imputed_val


machakos_clean, imputed_months, imputed_val = clean_machakos_series(
    machakos_raw, months
)
print("Imputed months (ValueError):", imputed_months)
print(f"Imputed value used: {imputed_val} mm")
print("Cleaned Machakos row:", machakos_clean)

Imputed months (ValueError): ['April', 'September']
Imputed value used: 68 mm
Cleaned Machakos row: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


**Task 2:** Combined $5 \times 12$ NumPy Array

In [6]:
R = np.array(clean_rows + [machakos_clean], dtype=int)
print("Array shape:", R.shape)
print("Array dtype:", R.dtype)

Array shape: (5, 12)
Array dtype: int64


**Task 3:** Annual totals per county (axis=1) and monthly averages across counties (axis=0)

In [7]:
annual_totals = R.sum(axis=1)
monthly_averages = R.mean(axis=0).round(1)

**Task 4:** Wettest month per county using np.argmax(axis=1) and zip()

In [8]:
wettest_indices = np.argmax(R, axis=1)
wettest_months = [months[idx] for idx in wettest_indices]

**Task 5:** Dry-month counts (< 50 mm) via boolean mask

In [9]:
dry_months = (R < 50).sum(axis=1)

**Task 6:** Best consecutive 2-month planting window via shifted array sum

In [10]:
two_month_sums = R[:, :-1] + R[:, 1:]
best_window_starts = np.argmax(two_month_sums, axis=1)
planting_windows = [
    f"{months[i]}-{months[i + 1]}" for i in best_window_starts
]

print("\nAnnual Totals (mm):")
for county, total in zip(counties, annual_totals):
  print(f"  {county:<10}: {total} mm")

print("\nMonthly Averages Across Counties (mm):")
for month, avg in zip(months, monthly_averages):
  print(f"  {month:<10}: {avg} mm")

print("\nCounty Summary:")
for county, wet, dry, window in zip(
    counties, wettest_months, dry_months, planting_windows
):
  print(
      f"  {county:<10} | Wettest: {wet:<9} | Dry (<50mm): {dry} | Window:"
      f" {window}"
  )


Annual Totals (mm):
  Nairobi   : 1045 mm
  Nakuru    : 916 mm
  Kisumu    : 1371 mm
  Mombasa   : 1075 mm
  Machakos  : 811 mm

Monthly Averages Across Counties (mm):
  January   : 47.0 mm
  February  : 47.6 mm
  March     : 91.0 mm
  April     : 164.4 mm
  May       : 171.2 mm
  June      : 67.6 mm
  July      : 56.6 mm
  August    : 58.2 mm
  September : 64.2 mm
  October   : 67.8 mm
  November  : 127.4 mm
  December  : 80.6 mm

County Summary:
  Nairobi    | Wettest: April     | Dry (<50mm): 4 | Window: April-May
  Nakuru     | Wettest: April     | Dry (<50mm): 3 | Window: April-May
  Kisumu     | Wettest: April     | Dry (<50mm): 0 | Window: April-May
  Mombasa    | Wettest: May       | Dry (<50mm): 2 | Window: April-May
  Machakos   | Wettest: November  | Dry (<50mm): 6 | Window: November-December


**Task 7:** `rainfall_summary.csv`

In [11]:
with open("rainfall_summary.csv", "w", newline="") as f:
  fieldnames = [
      "county",
      "annual_total_mm",
      "wettest_month",
      "dry_months",
      "planting_window",
  ]
  writer = csv.DictWriter(f, fieldnames=fieldnames)
  writer.writeheader()
  for county, total, wet, dry, window in zip(
      counties, annual_totals, wettest_months, dry_months, planting_windows
  ):
    writer.writerow({
        "county": county,
        "annual_total_mm": int(total),
        "wettest_month": wet,
        "dry_months": int(dry),
        "planting_window": window,
    })

# Quick verification read-back
with open("rainfall_summary.csv", newline="") as f:
  print("\nSaved rainfall_summary.csv:")
  print(f.read().strip())


Saved rainfall_summary.csv:
county,annual_total_mm,wettest_month,dry_months,planting_window
Nairobi,1045,April,4,April-May
Nakuru,916,April,3,April-May
Kisumu,1371,April,0,April-May
Mombasa,1075,May,2,April-May
Machakos,811,November,6,November-December


# Task 8: Markdown Sections

## Planting Advisory
Kisumu records the highest annual rainfall (1,371 mm) with zero dry months, making it the most reliable multi-season planting zone, whereas Machakos experiences the longest dry spell with 6 months below 50 mm. Nairobi, Nakuru, Kisumu, and Mombasa all peak around April–May, so SMS planting alerts for those four counties should target the April–May window. Machakos shifts later to a November–December peak window (278 mm combined), meaning farmers there should prepare land ahead of the short rains rather than matching coastal/lake timing. Because two Machakos readings (`April` and `September`) were imputed from the station's valid mean (`68` mm), field officers should verify local gauge logs before locking in seed distribution volumes.

## Data Quality Notes
The raw Machakos series contained unparseable entries (`"N/A"` in April and `""` in September) and whitespace padding (`" 20 "` in June). Invalid numeric strings were caught specifically with `except ValueError` and replaced with the rounded mean of the 10 valid Machakos entries (`68` mm) rather than silently dropping the row.